# 01 Reading Query Profiles

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Read a query's performance like a detective: find the operator where time is spent, check rows and bytes at each step, spot spill, skew and wasted scans, and connect what you see in the <b>query profile</b> (serverless and SQL warehouses) or <b>Spark UI</b> (classic) with the plan from <code>explain()</code>.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Optimization starts with measurement. Changing settings or code without reading the profile is guessing. Every lesson in this folder follows the same loop: <b>measure → find the bottleneck → fix one thing → measure again</b>. The query profile is the main measuring tool on Databricks.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What a query profile shows</b><br>
| Element | What to look at |<br>|---|---|<br>| <b>Operator graph</b> | Each operator (scan, filter, join, aggregate, exchange, sort) as a node, with arrows showing data flow |<br>| <b>Time spent per operator</b> | The top contributors. Optimize those first |<br>| <b>Rows</b> in and out | Explosions (rows grow after a join) or wasted work (many rows read, few kept) |<br>| <b>Bytes / files read</b> at scans | Whether pruning and data skipping worked: files pruned vs files read |<br>| <b>Shuffle / exchange</b> sizes | How much data moved between stages |<br>| <b>Spill</b> | Memory pressure: data written to disk during sort, aggregate or join |<br>| <b>Peak memory</b>, task time distribution | Skew (one task much longer) and memory hotspots |<br>| <b>Photon</b> operators | Whether the native engine ran the operator, or it fell back (UDFs) |
</div>

```
measure ──▶ biggest operator? ──▶ why? (rows, bytes, spill, skew, files) ──▶ fix ONE thing ──▶ measure again
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A report took 14 minutes. The profile showed 85% of the time in one scan reading 4,000 files and 1.2 TB, with "files pruned: 0", even though the query filtered on one week. The filter was <code>WHERE date_format(event_ts, 'yyyy-ww') = '2024-45'</code>, a function on the column, so no data skipping. Rewritten as a range on <code>event_ts</code>, the same scan read 60 files, and the query ran in 40 seconds. The profile pointed straight at the problem.
</div>

## How to open a profile

| Compute | Where |
|---|---|
| Serverless notebook | Under a cell's results → **See performance** → click the query → **query profile** |
| SQL warehouse / SQL editor | Under the results, or **Query History** → click a query → **See query profile** |
| Classic cluster | Spark UI → **SQL / DataFrame** tab → the query → graph and details, plus the **Stages** tab |

In the profile, switch between **graph** and **tree** views, sort operators by **time**, and click an operator to see its metrics.

## Setup: data for three specimen queries

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates an <code>events</code> Delta table (2 million rows, clustered by date) and a small <code>devices</code> dimension.<br><br>
<b>Why it matters</b><br>Each of the next sections runs a query with a typical profile signature. Run each one, open its profile, and compare what you see with the notes.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Row counts for the two tables.
</div>

In [0]:
DB = "workspace.optimization"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
import time
from pyspark.sql import functions as F

events_t, devices_t = f"{DB}.events", f"{DB}.devices"
(spark.range(2_000_000)
 .select(F.col("id").alias("event_id"),
         (F.col("id") % 20_000).alias("device_id"),
         F.expr("timestampadd(SECOND, cast(id * 13 % 7776000 AS INT), timestamp'2024-08-01 00:00:00')").alias("event_ts"),
         (F.rand(seed=1) * 100).alias("value"),
         F.sha2(F.col("id").cast("string"), 256).alias("payload"))
 .withColumn("event_date", F.to_date("event_ts"))
 .repartitionByRange(16, "event_date")
 .write.mode("overwrite").saveAsTable(events_t))
(spark.range(20_000).select(F.col("id").alias("device_id"),
                            F.element_at(F.array(*[F.lit(m) for m in ["A", "B", "C", "D"]]), (F.col("id") % 4 + 1).cast("int")).alias("model"))
 .write.mode("overwrite").saveAsTable(devices_t))
print("events:", spark.table(events_t).count(), "| devices:", spark.table(devices_t).count())

def run(label, df):
    start = time.time()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label:<45} {time.time() - start:6.2f}s  <- open this query's profile")

## 1. Specimen: a scan that can't skip data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Filters one week with a function wrapped around the column, then the same week as a range on the raw column.<br><br>
<b>What to look for in the profile</b><br>At the <b>scan</b> node: files read and bytes read. The function version reads every file. The range version reads a fraction of the files, because file min/max statistics on <code>event_ts</code> can rule the others out.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Two timings. In the profiles, the second scan reads far fewer files and bytes.
</div>

In [0]:
week_fn = spark.table(events_t).filter(F.date_format("event_ts", "yyyy-MM-dd") .between("2024-10-07", "2024-10-13"))
week_range = spark.table(events_t).filter((F.col("event_ts") >= "2024-10-07") & (F.col("event_ts") < "2024-10-14"))
run("1a. function on the column (no skipping)", week_fn.groupBy("device_id").count())
run("1b. range on the raw column (skipping)", week_range.groupBy("device_id").count())

## 2. Specimen: rows exploding after a join

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins events to a device table that accidentally contains duplicate keys, and compares with the deduplicated dimension.<br><br>
<b>What to look for in the profile</b><br><b>Rows</b> coming out of the join node: more rows out than events in means the dimension key isn't unique. Everything after the join then processes the inflated data, and totals are wrong.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The bad join outputs about 4 million rows (every event matched twice). The fixed join outputs 2 million.
</div>

In [0]:
dup_devices = spark.table(devices_t).unionByName(spark.table(devices_t))       # every key twice: a classic data bug
bad = spark.table(events_t).join(dup_devices, "device_id")
good = spark.table(events_t).join(spark.table(devices_t).dropDuplicates(["device_id"]), "device_id")
print("rows after bad join :", bad.count())
print("rows after good join:", good.count())
run("2a. join with duplicate dimension keys", bad.groupBy("model").agg(F.sum("value")))
run("2b. join with unique dimension keys", good.groupBy("model").agg(F.sum("value")))

## 3. Specimen: a big shuffle with wide rows

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Deduplicates events while carrying the wide <code>payload</code> column, then the same deduplication on only the needed columns.<br><br>
<b>What to look for in the profile</b><br><b>Exchange / shuffle</b> bytes written and read, and any <b>spill</b> on the aggregate. Carrying <code>payload</code> multiplies the shuffle size.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The lean version is faster, and its profile shows a much smaller shuffle.
</div>

In [0]:
run("3a. dedupe carrying payload", spark.table(events_t).dropDuplicates(["device_id", "event_date"]))
run("3b. dedupe with needed columns only", spark.table(events_t).select("device_id", "event_date", "value").dropDuplicates(["device_id", "event_date"]))

## 4. Profiles from code: query history metrics

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the metrics of your recent statements from <code>system.query.history</code>.<br><br>
<b>Why it matters</b><br>The same metrics as the profile, but queryable: compare runs before and after a fix, or find the most expensive queries of the week.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Your latest statements with duration, rows read, bytes read and spill, or a note if the table isn't accessible.
</div>

In [0]:
try:
    display(spark.sql("""
        SELECT start_time, total_duration_ms, read_rows, read_bytes, read_files, pruned_files,
               spilled_local_bytes, left(statement_text, 70) AS statement
        FROM system.query.history
        WHERE executed_by = current_user() AND start_time > current_timestamp() - INTERVAL 1 HOUR
        ORDER BY start_time DESC LIMIT 10
    """))
except Exception as e:
    print("system.query.history not accessible:", type(e).__name__)

## 5. Profile vs plan

| In the profile | In `explain("formatted")` |
|---|---|
| Scan node: files read / pruned, bytes | `Scan ... PushedFilters`, `PartitionFilters`, `ReadSchema` |
| Join node type | `BroadcastHashJoin`, `SortMergeJoin` (initial plan, before AQE) |
| Exchange node: shuffle bytes | `Exchange hashpartitioning(...)` |
| Time per operator, spill, rows | Not in the plan: only in the profile |
| AQE changes (coalesced partitions, skew splits, join switch) | Not in the initial plan |

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Rule of thumb</b><br>
Use <code>explain()</code> <b>before</b> running to predict, and the profile <b>after</b> running to measure. When they disagree, the profile (the final plan) is the truth.
</div>

In [0]:
week_range.groupBy("device_id").count().explain("formatted")

## My profile observation log

| Specimen | Time | Slowest operator | Rows in → out (key operator) | Files / bytes read | Shuffle bytes | Spill |
|---|---|---|---|---|---|---|
| 1a function filter | | | | | | |
| 1b range filter | | | | | | |
| 2a duplicate keys | | | | | | |
| 2b unique keys | | | | | | |
| 3a wide dedupe | | | | | | |
| 3b lean dedupe | | | | | | |

## Under the hood

```
During execution every operator updates metrics (rows, time, bytes, spill, peak memory)
  -> Spark SQL metrics (classic: Spark UI SQL tab)
  -> Databricks query profile (serverless / SQL warehouses), also stored in system.query.history
The profile shows the FINAL plan (after AQE), the plan shows the INITIAL one.
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> only executed queries have profiles. `explain()` doesn't run anything.

In [0]:
# Keep the tables for the next lessons in this folder (02-08 reuse them).
print("tables kept:", events_t, devices_t)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: optimizing the wrong part</b><br>
Look at time per operator first. A 2% operator isn't worth tuning.<br><br>
<b>⛔ Problem: the plan says one thing, the run another</b><br>AQE changed the plan at runtime. Read the final plan in the profile.<br><br>
<b>⛔ Problem: "files pruned: 0" on a filtered query</b><br>The filter can't use statistics (function on the column, a UDF) or the data isn't clustered on that column (lessons 03_10 and 03 of this folder).<br><br>
<b>⛔ Problem: rows grow after a join</b><br>Duplicate keys on the other side. Check key uniqueness.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How do you start investigating a slow query on Databricks?</b><br>
I open its query profile (or the Spark UI on classic compute), sort operators by time, and look at the top ones: rows in and out, bytes and files read, shuffle sizes, spill, and the task time distribution. That tells me whether the problem is reading too much, moving too much, exploding rows, skew, or memory.<br><br>

<b>🎤 2. What signals in a profile point to missing data skipping?</b><br>
A scan reading most files and bytes of a table even though the query filters on a column: few or zero files pruned. Usually the filter wraps the column in a function, uses a UDF, or the data isn't clustered on that column.<br><br>

<b>🎤 3. How do you spot a bad join in a profile?</b><br>
The join outputs far more rows than its inputs (duplicate keys), it's a sort-merge join with a huge shuffle where one side is small (missing broadcast), or one task takes much longer than the others (skew).<br><br>

<b>🎤 4. What is spill and where do you see it?</b><br>
Data written to disk because it didn't fit in memory during a sort, aggregation or join. It's reported per operator in the profile and per stage in the Spark UI.<br><br>

<b>🎤 5. What's the difference between <code>explain()</code> and a query profile?</b><br>
<code>explain()</code> shows the plan before execution, without metrics. The profile shows the final executed plan, including AQE changes, with real metrics per operator.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A data engineer wants to know which operator in a slow SQL query consumed the most time on a serverless SQL warehouse. Where should they look?</b><br>
A. <code>DESCRIBE HISTORY</code><br>
B. The query profile in Query History<br>
C. The cluster event log<br>
D. <code>SHOW TBLPROPERTIES</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A query profile shows a scan reading all 5,000 files of a table, although the query filters on one day. What is the most likely cause?</b><br>
A. The warehouse is too small<br>
B. The filter can't use data skipping (for example a function on the column), or the data isn't clustered by that column<br>
C. Photon is enabled<br>
D. The table has too few columns<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Fill in the observation log for the six specimen runs</li><li>Write a query on <code>events</code> that counts events per model for October, as efficiently as you can, and record its profile</li><li>Make it deliberately worse in one way (a UDF filter, or no column pruning) and compare the profiles</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
from pyspark.sql.functions import udf

# 2. Efficient: range filter on the raw column, broadcast the small dimension, only needed columns
efficient = (spark.table(events_t)
             .filter((F.col("event_ts") >= "2024-10-01") & (F.col("event_ts") < "2024-11-01"))
             .select("device_id")
             .join(F.broadcast(spark.table(devices_t)), "device_id")
             .groupBy("model").count())
run("efficient", efficient)

# 3. Worse: a Python UDF filter can't be pushed down or skip files
@udf("boolean")
def in_october(year_month):
    return year_month == "2024-10"

worse = spark.table(events_t).filter(in_october(F.date_format("event_ts", "yyyy-MM"))).join(spark.table(devices_t), "device_id").groupBy("model").count()
run("worse (UDF filter)", worse)
print("same result:", sorted(efficient.collect()) == sorted(worse.collect()))

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Measure first: the query profile (serverless, SQL warehouses) or Spark UI (classic)</li><li>Look at time per operator, rows in and out, files and bytes read, shuffle sizes, spill, and task skew</li><li>Typical signatures: no file pruning, row explosion after joins, huge shuffles, spill, one slow task</li><li><code>explain()</code> predicts. The profile measures, and shows AQE's final plan</li><li>Fix one thing at a time, and measure again</li></ul>
</div>

| Symptom in the profile | Likely cause | Lesson |
|---|---|---|
| Scan reads everything | No skipping or pruning | 03, Delta 10 |
| Rows explode after a join | Duplicate keys | 05 |
| Big exchange bytes | Wide rows, unnecessary shuffle | 06 |
| Spill | Large partitions, skew | 04 |
| One slow task | Skew | 02 |

## Git commit

```
git add 05_optimization/01_reading_query_profiles.ipynb
git commit -m "add 05_01 reading query profiles notebook"
```